In [ ]:
# ============================================================
# Import e setup
# ============================================================
import sys
import os
import yaml
import numpy as np
import pandas as pd
import torch

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

from src.datasets.gqa_graph_dataset import GQAGraphDataset
from src.models.gnn import GCNEmbeddingNet, GINEEmbeddingNet
from src.utils.gnnExplainer import (
    GCNWrapper, GINEWrapper,
    build_explainer,
    get_node_names,
    plot_comparison,
)


In [ ]:

# ============================================================
# Caricamento GCN
# ============================================================
config_path = os.path.join(BASE_DIR, "experiments", "conf", "gcn_config.yaml")
with open(config_path) as f:
    cfg_gcn = yaml.safe_load(f)

device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt_path  = os.path.join(BASE_DIR, cfg_gcn["training"]["checkpoint"])
checkpoint = torch.load(ckpt_path, map_location=device)

node_vocab_gcn = checkpoint["node_vocab"]
rel_vocab_gcn  = checkpoint["rel_vocab"]

gcn_model = GCNEmbeddingNet(
    num_node_types=len(node_vocab_gcn) + 1,
    emb_dim=cfg_gcn["model"]["emb_dim"],
    hidden_dim=cfg_gcn["model"]["hidden_dim"],
    num_layers=cfg_gcn["model"]["num_layers"],
    dropout=cfg_gcn["model"]["dropout"],
    use_bbox=cfg_gcn["model"]["use_bbox"],
    node_emb_dim=cfg_gcn["model"]["node_emb_dim"],
).to(device)

gcn_model.load_state_dict(checkpoint["model_state_dict"])
gcn_model.eval()
print(f"GCN caricato da: {ckpt_path}")

gcn_wrapper  = GCNWrapper(gcn_model).to(device)
explainer_gcn = build_explainer(gcn_wrapper)


In [ ]:

# ============================================================
# Caricamento GINE
# ============================================================
config_path = os.path.join(BASE_DIR, "experiments", "conf", "gine_config.yaml")
with open(config_path) as f:
    cfg_gine = yaml.safe_load(f)

ckpt_path  = os.path.join(BASE_DIR, cfg_gine["training"]["checkpoint"])
checkpoint = torch.load(ckpt_path, map_location=device)

node_vocab_gine = checkpoint["node_vocab"]
rel_vocab_gine  = checkpoint["rel_vocab"]

gine_model = GINEEmbeddingNet(
    num_node_types=len(node_vocab_gine) + 1,
    edge_vocab_size=len(rel_vocab_gine),
    emb_dim=cfg_gine["model"]["emb_dim"],
    hidden_dim=cfg_gine["model"]["hidden_dim"],
    num_layers=cfg_gine["model"]["num_layers"],
    dropout=cfg_gine["model"]["dropout"],
    use_bbox=cfg_gine["model"]["use_bbox"],
    node_emb_dim=cfg_gine["model"]["node_emb_dim"],
).to(device)

gine_model.load_state_dict(checkpoint["model_state_dict"])
gine_model.eval()
print(f"GINE caricato da: {ckpt_path}")

gine_wrapper  = GINEWrapper(gine_model).to(device)
explainer_gine = build_explainer(gine_wrapper)


In [ ]:

# ============================================================
# Dataset di validazione
# ============================================================
df_train = pd.read_csv(os.path.join(BASE_DIR, cfg_gcn["data"]["train_csv"]))
df_val   = pd.read_csv(os.path.join(BASE_DIR, cfg_gcn["data"]["val_csv"]))

all_labels = df_train["labels"].unique()
label2idx  = {l: i for i, l in enumerate(all_labels)}
idx2label  = {i: l for l, i in label2idx.items()}

df_train["labels"] = df_train["labels"].map(label2idx)
df_val["labels"]   = df_val["labels"].map(label2idx)

val_dataset_gcn = GQAGraphDataset(
    df=df_val, label_col="labels",
    node_vocab=node_vocab_gcn, rel_vocab=rel_vocab_gcn,
    label2idx=label2idx, idx2label=idx2label,
    use_bbox=cfg_gcn["model"]["use_bbox"],
)

val_dataset_gine = GQAGraphDataset(
    df=df_val, label_col="labels",
    node_vocab=node_vocab_gine, rel_vocab=rel_vocab_gine,
    label2idx=label2idx, idx2label=idx2label,
    use_bbox=cfg_gine["model"]["use_bbox"],
)

IMAGE_DIR = os.path.join(BASE_DIR, "data", "images")
print(f"Val dataset size: {len(val_dataset_gcn)}")


In [ ]:

# ============================================================
# Confronto GCN vs GINE su 3 campioni
# ============================================================
np.random.seed(42)
sample_indices = np.random.choice(len(val_dataset_gcn), size=3, replace=False)

for idx in [int(i) for i in sample_indices]:
    plot_comparison(
        idx=idx,
        df_val=df_val,
        image_dir=IMAGE_DIR,
        idx2label=idx2label,
        val_dataset_gcn=val_dataset_gcn,
        val_dataset_gine=val_dataset_gine,
        node_vocab_gcn=node_vocab_gcn,
        node_vocab_gine=node_vocab_gine,
        rel_vocab_gine=rel_vocab_gine,
        explainer_gcn=explainer_gcn,
        explainer_gine=explainer_gine,
        device=device,
        topk=8,
    )